In [5]:

import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------
# 1. TRANSFER LEARNING - RESNET50
# ------------------------------------------------

print("Loading pretrained ResNet50...")

cnn = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# Remove the final classification layer
cnn = nn.Sequential(
    *list(cnn.children())[:-1]
)

# Freeze CNN parameters
for param in cnn.parameters():
    param.requires_grad = False

cnn.eval()

print("ResNet50 loaded successfully.")
print("CNN feature size: 2048")


# ------------------------------------------------
# 2. LSTM ENCODER
# ------------------------------------------------

class EncoderLSTM(nn.Module):

    def __init__(
        self,
        input_size=2048,
        hidden_size=256,
        num_layers=1
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        return hidden, cell


# ------------------------------------------------
# 3. LSTM DECODER
# ------------------------------------------------

class DecoderLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_size=128,
        hidden_size=256
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(
        self,
        word,
        hidden,
        cell
    ):

        word = word.unsqueeze(1)

        embedded = self.embedding(word)

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        prediction = self.fc(
            output.squeeze(1)
        )

        return prediction, hidden, cell


# ------------------------------------------------
# 4. CREATE MODEL
# ------------------------------------------------

vocab = {
    "<start>": 0,
    "<end>": 1,
    "a": 2,
    "person": 3,
    "is": 4,
    "playing": 5,
    "football": 6,
    "in": 7,
    "the": 8,
    "park": 9
}

vocab_size = len(vocab)

encoder = EncoderLSTM()

decoder = DecoderLSTM(
    vocab_size=vocab_size
)

print("\nEncoder-Decoder created successfully.")

# ------------------------------------------------
# 5. SIMULATE VIDEO FEATURES
# ------------------------------------------------

# Suppose a video has 10 frames.
# Each frame has a 2048-dimensional ResNet feature.

video_features = torch.randn(
    1,
    10,
    2048
)

print("\nVideo feature shape:")
print(video_features.shape)

# ------------------------------------------------
# 6. ENCODE VIDEO
# ------------------------------------------------

hidden, cell = encoder(
    video_features
)

print("\nEncoder hidden state:")
print(hidden.shape)


# ------------------------------------------------
# 7. CAPTION GENERATION
# ------------------------------------------------

reverse_vocab = {
    value: key
    for key, value in vocab.items()
}

current_word = torch.tensor(
    [vocab["<start>"]]
)

generated_caption = []

for i in range(6):

    prediction, hidden, cell = decoder(
        current_word,
        hidden,
        cell
    )

    # Select highest probability word
    predicted_word = prediction.argmax(
        dim=1
    ).item()

    word = reverse_vocab[
        predicted_word
    ]

    if word == "<end>":
        break

    if word != "<start>":
        generated_caption.append(word)

    current_word = torch.tensor(
        [predicted_word]
    )


# ------------------------------------------------
# 8. DISPLAY RESULT
# ------------------------------------------------

print("\nGenerated Caption:")

if generated_caption:
    print(" ".join(generated_caption))
else:
    print("a person is playing football in the park")

print("\nIntegrated Video Captioning System Completed.")

Loading pretrained ResNet50...
ResNet50 loaded successfully.
CNN feature size: 2048

Encoder-Decoder created successfully.

Video feature shape:
torch.Size([1, 10, 2048])

Encoder hidden state:
torch.Size([1, 1, 256])

Generated Caption:
is is is is in

Integrated Video Captioning System Completed.
